# Jainish — BillSum Week 3 learning-rate selection

New implementation: GPU validation is pending. Run cells individually. Start with debug checks; keep Weeks 1 and 2 unchanged.

Protocol: eight seed-42 trials, three epochs each, tuning ROUGE-L selection. No final-test scoring. Read `docs/billsum_week3.md` before full trials. A local checkpoint is lost if the session disk disappears: persist weights AND reports before stopping a session.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Execute before importing torch.
from pathlib import Path
import subprocess, sys
project = Path("/kaggle/working/lora-on-a-budget")
branch = "jainish/week3-billsum-lr-selection"
if not project.exists():
    subprocess.run(["git", "clone", "--branch", branch,
                    "https://github.com/devjp99/lora-on-a-budget.git", str(project)], check=True)
else:
    current = subprocess.check_output(["git", "branch", "--show-current"], cwd=project, text=True).strip()
    if current != branch:
        raise RuntimeError(f"Existing folder uses {current}; review it before continuing.")
os.chdir(project)
print(subprocess.check_output(["git", "log", "-1", "--format=%h %s"], text=True))
assert (project / "scripts/train_billsum.py").exists(), "Upload and commit the Week 3 package first."


In [ ]:
import importlib.metadata as metadata
import platform
print("Python:", platform.python_version())
print("torch:", metadata.version("torch"))
assert metadata.version("torch") == "2.11.0+cu130", "Stop: review changed Kaggle torch before installing."
assert platform.python_version_tuple()[:2] == ("3", "13"), "Stop: review Python version."
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-billsum-week3.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)


**Restart the kernel after installation.** Then run the next cell, which restores the directory and variables. Do not rerun installation unless the environment needs restoring.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from pathlib import Path
import subprocess, sys, json
from datetime import datetime, timezone
project = Path("/kaggle/working/lora-on-a-budget")
os.chdir(project)
logs = project / "results/billsum/week3/setup"
logs.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
def logged(command, label):
    path = logs / f"{label}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.txt"
    with path.open("w") as output:
        process = subprocess.Popen(command, cwd=project, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="")
            output.write(line); output.flush()
        result = process.wait()
    print("Exit code:", result, "Log:", path)
    if result:
        raise RuntimeError(f"{label} exited {result}; preserve and inspect the log.")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "unavailable")
print("Visible GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
assert "T4" in torch.cuda.get_device_name(0)
logged([sys.executable, "-m", "pip", "freeze"], "pip_freeze")
logged(["git", "rev-parse", "HEAD"], "git_commit")
logged([sys.executable, "-m", "pytest", "-q", "tests/test_billsum.py", "tests/test_billsum_week3.py"], "tests")
logged([sys.executable, "scripts/make_billsum_manifest.py", "--verify-existing"], "manifests")


## Run one debug check

Start with full FT. After it passes, repeat for LoRA, DoRA and QLoRA, changing METHOD, LR and RUN_ID. Each debug check trains 32 examples for one epoch and evaluates two examples. These runs do not count as LR trials.

For a resume check, use a new LoRA debug run ID and set STOP_AFTER_UPDATES=1. An interrupted error is expected. Then set RESUME=True and STOP_AFTER_UPDATES=None and execute the same run ID again.

Only after debug checks, environment/config review and time-budget review should PHASE become `lr_search`. Use a new unique run ID for each full trial.

In [ ]:
METHOD = "full_ft"
LR = 1e-5
PHASE = "debug"
RUN_ID = "debug-full-ft-01"
RESUME = False
STOP_AFTER_UPDATES = None

command = [sys.executable, "-u", "scripts/train_billsum.py",
           "--config", f"configs/billsum/{METHOD}.yaml",
           "--lr", str(LR), "--seed", "42", "--phase", PHASE, "--run-id", RUN_ID]
if RESUME:
    command.append("--resume")
if STOP_AFTER_UPDATES is not None:
    command.extend(["--stop-after-updates", str(STOP_AFTER_UPDATES)])
logged(command, RUN_ID)


In [ ]:
report_path = project / "results/billsum/training" / f"{RUN_ID}.json"
report = json.loads(report_path.read_text())
for key in ["phase", "status", "successful_optimizer_steps", "best_epoch",
            "best_rougeL", "reload_passed", "reload_max_abs_diff", "best_checkpoint"]:
    print(key, ":", report.get(key))


## Preserve files before ending the session

Keep `results/billsum/training/`, `results/billsum/week3/`, and the entire `checkpoints/billsum/training/RUN_ID/` folder. The checkpoint folder can be several GB. Save notebook outputs and download/copy those files to durable storage. Reports alone cannot resume training. Do not upload model checkpoints to GitHub.

## Full search grid (after debug validation)

| Method | LR candidates | Seed | Epochs |
|---|---|---|---|
| full_ft | 1e-5, 3e-5 | 42 | 3 |
| lora | 1e-4, 3e-4 | 42 | 3 |
| dora | 1e-4, 3e-4 | 42 | 3 |
| qlora | 1e-4, 3e-4 | 42 | 3 |

The eight runs are separate. Stop and preserve any failure. Do not change shared settings for just one method.

## Select learning rates

Run this cell only after all eight full trials are completed. It rejects incomplete or incomparable evidence.

In [ ]:
logged([sys.executable, "scripts/select_lr_billsum.py"], "selection")
